# Tutorial: Explicit Scopes for QSAR and Docking

Audience:
- AMDockVS developers.

Prerequisites:
- AMDockVS and MolSuite can be imported in the environment.
- Sample data is available in data/docking/.

Learning goals:
- Verify that qsar.compute_descriptors() accepts MoleculeScope and MoleculeSetRef.
- Verify that docking.check_required() accepts molecule scopes.
- Confirm that preflight reports unprepared ligands and receptors before submitting Vina.

## Outline

1. Set up
2. Create a project and import a ligand/receptor
3. Create explicit scopes
4. Run QSAR with ligand_set
5. Check docking prerequisites for the scopes

In [ ]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate repo root")


REPO_ROOT = find_repo_root(Path.cwd())
AMDOCK_SRC = REPO_ROOT / "src"
MOLSUITE_SRC = REPO_ROOT.parent / "MolSuite" / "src"
for path in (AMDOCK_SRC, MOLSUITE_SRC):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

from amdockvs.runtime import AMDockVSRuntime

TMP_ROOT = REPO_ROOT / "notebooks" / "_tmp"
TMP_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = Path(tempfile.mkdtemp(prefix="05_explicit_sets_", dir=TMP_ROOT))
LIGAND_FILE = REPO_ROOT / "data" / "docking" / "1iep_ligand.sdf"
RECEPTOR_FILE = REPO_ROOT / "data" / "docking" / "1iep_receptor.pdbqt"
assert LIGAND_FILE.exists(), LIGAND_FILE
assert RECEPTOR_FILE.exists(), RECEPTOR_FILE

runtime = AMDockVSRuntime()
runtime.create_project(
    name=f"explicit-sets-{PROJECT_DIR.name}",
    folder=PROJECT_DIR,
    description="Notebook smoke test for explicit set contract.",
)

{
    "project_dir": str(PROJECT_DIR),
    "ligand_file": str(LIGAND_FILE),
    "receptor_file": str(RECEPTOR_FILE),
}


In [ ]:
import_job_ids = [
    *runtime.loader.load_ligands([LIGAND_FILE], executor_name="thread"),
    *runtime.loader.load_receptors([RECEPTOR_FILE], executor_name="thread"),
]
import_status = runtime.wait_for_jobs(import_job_ids, timeout_s=180.0, poll_s=0.2)
assert all(row.status == "completed" for row in import_status.values()), import_status
import_status


In [ ]:
ligand_scope = runtime.molecules.filter(
    runtime.molecules.all(),
    filters={"primary_role": "ligand", "input_format": "sdf"},
)
receptor_scope = runtime.molecules.select(role="receptor")

ligand_ids = list(runtime.molecules.stream_ids(ligand_scope))
receptor_ids = list(runtime.molecules.stream_ids(receptor_scope))

assert ligand_ids, "Expected ligand ids"
assert receptor_ids, "Expected receptor ids"

{
    "ligand_ids": ligand_ids,
    "receptor_ids": receptor_ids,
}


In [ ]:
descriptor_job_id = runtime.qsar.compute_descriptors(
    molecule_set=ligand_scope,
    batch_size=16,
    executor_name="process",
    depends_on=import_job_ids,
)
descriptor_status = runtime.wait_for_job(descriptor_job_id, poll_s=0.2)
assert descriptor_status.status == "completed", descriptor_status

descriptors = runtime.qsar.list_descriptors()
assert descriptors, "Expected descriptor rows"

{
    "descriptor_job_id": descriptor_job_id,
    "descriptor_rows": len(descriptors),
    "first_descriptor_status": descriptors[0].status,
}


In [ ]:
requirement_check = runtime.docking.check_required(
    method="vina",
    ligand_set=ligand_scope,
    receptor_set=receptor_scope,
)

assert requirement_check["ready"] is False, requirement_check
assert requirement_check["missing"]["ligands_prepared_for_vina"], requirement_check
assert requirement_check["missing"]["receptors_prepared_for_vina"], requirement_check
assert requirement_check["missing"]["receptor_grids_vina"], requirement_check

{
    "ready": requirement_check["ready"],
    "counts": requirement_check["counts"],
    "missing": requirement_check["missing"],
}
